# Agent Evaluation con LangSmith + agentevals

En este notebook aprenderemos a evaluar agentes usando:
- **agentevals**: librería oficial de LangChain con evaluadores prearmados
- **LangSmith**: plataforma para registrar y visualizar experimentos
- **NVIDIA NIM**: como proveedor del LLM

## ¿Por qué evaluar agentes?

A diferencia de una función Python, un agente es no-determinístico:
- El mismo input puede producir trayectorias diferentes
- Las tools pueden fallar silenciosamente
- El modelo puede alucinar o entrar en loops

Necesitamos métricas para saber si nuestro agente está funcionando bien.

## 0. Instalación

In [ ]:
!pip install langgraph langchain-openai langchain-core agentevals langsmith duckduckgo-search python-dotenv

## 1. Setup: API Keys y configuración

In [1]:
import os
import json
import time
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.errors import GraphRecursionError
from langsmith import Client
from typing import Annotated
from typing_extensions import TypedDict

load_dotenv()

# NVIDIA NIM via OpenAI-compatible API
MODEL = "meta/llama-3.1-70b-instruct"

llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
    model=MODEL,
)

print(f"LLM: {MODEL}")
print(f"LangSmith tracing: {os.environ.get('LANGCHAIN_TRACING_V2', 'not set')}")
print(os.environ.get("LANGSMITH_API_KEY")[:10]) # debe mostrar "lsv2_..."

LLM: meta/llama-3.1-70b-instruct
LangSmith tracing: true
lsv2_pt_97


## 2. Agente simple con DDGS

Usamos un agente sencillo con una sola tool de búsqueda web.
El foco de esta clase es la **evaluación**, no el agente en sí.

In [2]:
from ddgs import DDGS

@tool
def web_search(query: str) -> str:
    """Busca información actualizada en la web usando DuckDuckGo."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No se encontraron resultados."
        return "\n\n".join(f"**{r['title']}**\n{r['body']}" for r in results)
    except Exception as e:
        return f"Error al buscar: {e}" 

In [3]:
TOOLS = [web_search]
llm_with_tools = llm.bind_tools(TOOLS, parallel_tool_calls=False)

SYSTEM_PROMPT = SystemMessage(
    content="Eres un asistente útil con acceso a búsqueda web. "
            "Usa web_search cuando necesites información actualizada. "
            "Responde de forma concisa y directa."
)


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


def call_model(state: AgentState) -> AgentState:
    messages = [SYSTEM_PROMPT] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END


graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", call_model)
graph_builder.add_node("tools", ToolNode(TOOLS))
graph_builder.set_entry_point("agent")
graph_builder.add_conditional_edges("agent", should_continue)
graph_builder.add_edge("tools", "agent")

agent = graph_builder.compile()
print("Agente compilado ✅")

Agente compilado ✅


Prueba rápida del agente:

In [4]:
result = agent.invoke(
    {"messages": [HumanMessage(content="¿Cuántas lunas tiene Marte?")]},
    config={"recursion_limit": 10}
)
print(result["messages"][-1].content)

Marte tiene dos lunas, que se llaman Fobos y Deimos.


## 3. Dataset de evaluación

Un dataset de eval tiene:
- **inputs**: la pregunta que le hacemos al agente
- **expected_tools**: qué tools esperamos que use (opcional)
- **reference_outputs**: la respuesta correcta (para LLM-as-judge)

In [5]:
eval_dataset = [
    {
        "question": "¿Cuántas lunas tiene Marte?",
        "expected_answer": "Marte tiene 2 lunas: Fobos y Deimos",
        "expected_tools": [],
        "expected_content": [],       # no aplica: no usa web_search
    },
    {
        "question": "¿Quién es el CEO actual de NVIDIA?",
        "expected_answer": "Jensen Huang es el CEO de NVIDIA",
        "expected_tools": ["web_search"],
        "expected_content": ["Jensen Huang", "NVIDIA", "CEO"],
    },
    {
        "question": "¿Cuál es la capital de Colombia?",
        "expected_answer": "La capital de Colombia es Bogotá",
        "expected_tools": [],
        "expected_content": [],
    },
    {
        "question": "¿Cuándo fue fundada la Universidad EAFIT?",
        "expected_answer": "La Universidad EAFIT fue fundada en 1960",
        "expected_tools": ["web_search"],
        "expected_content": ["EAFIT", "1960", "Medellín"],
    },
    {
        "question": "¿Cuál es la fórmula del agua?",
        "expected_answer": "La fórmula del agua es H2O",
        "expected_tools": [],
        "expected_content": [],
    },
]

print(f"Dataset: {len(eval_dataset)} preguntas")

Dataset: 5 preguntas


## 4. Helper: extraer trayectoria del agente

Para evaluar con agentevals necesitamos la trayectoria completa
(todos los mensajes incluyendo tool calls), no solo la respuesta final.

In [6]:
def run_agent_and_get_trajectory(question: str) -> dict:
    """
    Corre el agente y retorna:
    - answer: respuesta final
    - trajectory: lista de mensajes en formato OpenAI (para agentevals)
    - tools_called: nombres de tools que se llamaron
    """
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 10}
    )

    messages = result["messages"]
    trajectory = []
    tools_called = []

    trajectory.append({"role": "user", "content": question})

    for msg in messages:
        msg_type = type(msg).__name__

        if msg_type == "AIMessage":
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                trajectory.append({
                    "role": "assistant",
                    "content": msg.content or "",
                    "tool_calls": [
                        {"function": {"name": tc["name"], "arguments": json.dumps(tc["args"])}}
                        for tc in msg.tool_calls
                    ]
                })
                tools_called.extend([tc["name"] for tc in msg.tool_calls])
            else:
                trajectory.append({"role": "assistant", "content": msg.content})

        elif msg_type == "ToolMessage":
            trajectory.append({"role": "tool", "content": str(msg.content)})

    return {
        "answer": messages[-1].content,
        "trajectory": trajectory,
        "tools_called": tools_called,
    }


# Prueba
test = run_agent_and_get_trajectory("¿Cuántas lunas tiene Marte?")
print("Respuesta:", test["answer"])
print("Tools llamadas:", test["tools_called"])
print("Pasos en trayectoria:", len(test["trajectory"]))

Respuesta: Marte tiene dos lunas, que se llaman Fobos y Deimos.
Tools llamadas: ['web_search']
Pasos en trayectoria: 4


## 5. Evaluador 1: Tool use (determinístico)

El evaluador más simple: verificar si el agente llamó las tools correctas.
**No necesita LLM** — es una comparación directa.

✅ Ventaja: rápido, barato, reproducible  
❌ Desventaja: no evalúa calidad de la respuesta

In [7]:
def evaluate_tool_use(tools_called: list, expected_tools: list) -> dict:
    called_set = set(tools_called)
    expected_set = set(expected_tools)

    if expected_set == set():
        score = 1.0 if called_set == set() else 0.0
        comment = "✅ No llamó tools innecesarias" if score == 1.0 else f"❌ Llamó tools cuando no debía: {called_set}"
    else:
        score = 1.0 if expected_set.issubset(called_set) else 0.0
        comment = "✅ Llamó las tools correctas" if score == 1.0 else f"❌ Esperadas: {expected_set}, llamadas: {called_set}"

    return {"score": score, "comment": comment}


# Prueba
print(evaluate_tool_use([], []))                          # ✅
print(evaluate_tool_use(["web_search"], ["web_search"]))  # ✅
print(evaluate_tool_use([], ["web_search"]))              # ❌

{'score': 1.0, 'comment': '✅ No llamó tools innecesarias'}
{'score': 1.0, 'comment': '✅ Llamó las tools correctas'}
{'score': 0.0, 'comment': "❌ Esperadas: {'web_search'}, llamadas: set()"}


## 6. Evaluador 2: Retrieval quality (determinístico)

Evalúa si los resultados de `web_search` contienen la información
necesaria para responder la pregunta.

Opera sobre los mensajes `role: tool` de la trayectoria —
lo que la búsqueda realmente devolvió.

Métrica: **Recall@expected_content**
¿Qué fracción de los términos esperados aparecieron en los resultados?

In [8]:
def evaluate_retrieval(trajectory: list, expected_content: list) -> dict:
    """
    Evalúa la calidad del retrieval verificando si los resultados
    de web_search contienen los términos esperados.
    Solo aplica a preguntas con expected_content no vacío.
    """
    if not expected_content:
        return {"score": None, "comment": "N/A: no requiere búsqueda"}

    tool_results = " ".join(
        msg["content"] for msg in trajectory if msg["role"] == "tool"
    ).lower()

    if not tool_results:
        return {"score": 0.0, "comment": "❌ No se llamó ninguna tool"}

    found = [term for term in expected_content if term.lower() in tool_results]
    score = len(found) / len(expected_content)

    return {
        "score": score,
        "found": found,
        "missing": [t for t in expected_content if t.lower() not in tool_results],
        "comment": f"{len(found)}/{len(expected_content)} términos encontrados: {found}",
    }


# Prueba con CEO de NVIDIA
sample = eval_dataset[1]
agent_result = run_agent_and_get_trajectory(sample["question"])
retrieval_result = evaluate_retrieval(agent_result["trajectory"], sample["expected_content"])
print("Retrieval score:", retrieval_result)

Retrieval score: {'score': 1.0, 'found': ['Jensen Huang', 'NVIDIA', 'CEO'], 'missing': [], 'comment': "3/3 términos encontrados: ['Jensen Huang', 'NVIDIA', 'CEO']"}


## 7. Evaluador 3: Trajectory match (agentevals)

agentevals compara la trayectoria del agente contra una trayectoria de referencia.
Evalúa si el **proceso** fue correcto, no solo la respuesta final.

Modos disponibles:
| Modo | Descripción |
|---|---|
| `strict` | Mismo orden, mismas tools con mismos args |
| `unordered` | Mismas tools, cualquier orden |
| `subset` | El agente llamó **al menos** las tools esperadas |
| `superset` | El agente no llamó tools fuera de las esperadas |

In [9]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

trajectory_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="subset",   # el agente debe llamar AL MENOS las tools esperadas
    tool_args_match_mode="ignore",    # no comparamos argumentos exactos, solo nombres
)

def build_reference_trajectory(question: str, expected_tools: list, expected_answer: str) -> list:
    traj = [{"role": "user", "content": question}]
    if expected_tools:
        traj.append({
            "role": "assistant",
            "content": "",
            "tool_calls": [{"function": {"name": t, "arguments": "{}"}} for t in expected_tools]
        })
        traj.append({"role": "tool", "content": "resultado de búsqueda"})
    traj.append({"role": "assistant", "content": expected_answer})
    return traj


# Prueba con CEO de NVIDIA
sample = eval_dataset[1]
agent_result = run_agent_and_get_trajectory(sample["question"])
reference = build_reference_trajectory(sample["question"], sample["expected_tools"], sample["expected_answer"])

traj_result = trajectory_evaluator(outputs=agent_result["trajectory"], reference_outputs=reference)
print("Trajectory match:", traj_result)

Trajectory match: {'key': 'trajectory_subset_match', 'score': True, 'comment': None, 'metadata': None}


## 7. Evaluador 3: LLM-as-judge (agentevals)

El evaluador más sofisticado: usa un LLM para juzgar si la trayectoria
fue razonable, **sin necesitar una trayectoria de referencia exacta**.

✅ Ventaja: flexible, no necesita ground truth  
❌ Desventaja: costoso, no determinístico

In [10]:
from agentevals.trajectory.llm import (
    create_trajectory_llm_as_judge,
    TRAJECTORY_ACCURACY_PROMPT,
)

judge_llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
    model=MODEL,
)

llm_judge = create_trajectory_llm_as_judge(
    prompt=TRAJECTORY_ACCURACY_PROMPT,
    judge=judge_llm,   # pasas el cliente directamente
)

# Prueba con lunas de Marte
sample = eval_dataset[0]
agent_result = run_agent_and_get_trajectory(sample["question"])
judge_result = llm_judge(outputs=agent_result["trajectory"])
print("LLM-as-judge:", judge_result)

LLM-as-judge: {'key': 'trajectory_accuracy', 'score': True, 'comment': "The goal of the trajectory is to answer the user's question about the number of moons Mars has. The input is a simple question in Spanish, and the output is a direct answer to that question.", 'metadata': None}


In [11]:
from pydantic import BaseModel

class EvalScores(BaseModel):
    correctness: float
    efficiency: float
    faithfulness: float
    comment: str

CUSTOM_EVAL_PROMPT = """You are an expert evaluator of AI agent trajectories.
Evaluate the following agent trajectory on THREE dimensions.

<Trajectory>
{outputs}
</Trajectory>

Score each dimension from 0.0 to 1.0:

1. **Correctness**: Is the final answer factually correct and complete?
   - 1.0: Correct and complete | 0.5: Partially correct | 0.0: Wrong or missing

2. **Efficiency**: Did the agent use the minimum tools necessary?
   - 1.0: Only called tools when truly needed
   - 0.5: Called some unnecessary tools
   - 0.0: Wrong tool usage pattern

3. **Faithfulness**: Is the final answer grounded in the tool results?
   - 1.0: Fully based on retrieved data
   - 0.5: Mixes retrieved data with assumptions
   - 0.0: Ignores tool results or hallucinates

Respond ONLY with JSON:
{{
  "correctness": <float 0.0-1.0>,
  "efficiency": <float 0.0-1.0>,
  "faithfulness": <float 0.0-1.0>,
  "comment": "<brief explanation in Spanish>"
}}"""


def multidim_judge(trajectory: list) -> dict:
    structured_judge = llm.with_structured_output(EvalScores)
    prompt = CUSTOM_EVAL_PROMPT.format(
        outputs=json.dumps(trajectory, indent=2, ensure_ascii=False)
    )
    result: EvalScores = structured_judge.invoke([HumanMessage(content=prompt)])
    return {
        "correctness": result.correctness,
        "efficiency": result.efficiency,
        "faithfulness": result.faithfulness,
        "comment": result.comment,
    }

## 8. Loop completo de evaluación

Corremos los 4 evaluadores sobre todo el dataset.

In [12]:
def run_full_eval(dataset: list, sleep_between: int = 8) -> list:
    results = []

    for i, item in enumerate(dataset):
        print(f"\n[{i+1}/{len(dataset)}] {item['question'][:50]}...")
        try:
            agent_result = run_agent_and_get_trajectory(item["question"])

            tool_eval = evaluate_tool_use(agent_result["tools_called"], item["expected_tools"])

            retrieval_eval = evaluate_retrieval(
                agent_result["trajectory"], item["expected_content"]
            )

            reference = build_reference_trajectory(
                item["question"], item["expected_tools"], item["expected_answer"]
            )
            traj_eval = trajectory_evaluator(
                outputs=agent_result["trajectory"], reference_outputs=reference
            )

            judge_eval = multidim_judge(agent_result["trajectory"])

            results.append({
                "question": item["question"],
                "answer": agent_result["answer"],
                "tools_called": agent_result["tools_called"],
                "expected_tools": item["expected_tools"],
                "tool_use_score": tool_eval["score"],
                "tool_use_comment": tool_eval["comment"],
                "retrieval_score": retrieval_eval["score"],
                "retrieval_comment": retrieval_eval["comment"],
                "trajectory_match": traj_eval["score"],
                "correctness": judge_eval["correctness"],
                "efficiency": judge_eval["efficiency"],
                "faithfulness": judge_eval["faithfulness"],
                "llm_judge_comment": judge_eval["comment"],
                "status": "ok",
            })

            retrieval_str = f"{retrieval_eval['score']:.2f}" if retrieval_eval["score"] is not None else "N/A"
            print(f"  tool={tool_eval['score']:.1f} | retrieval={retrieval_str} | traj={float(traj_eval['score']):.1f} | correct={judge_eval['correctness']:.1f} | efficient={judge_eval['efficiency']:.1f} | faithful={judge_eval['faithfulness']:.1f}")

        except GraphRecursionError:
            results.append({"question": item["question"], "status": "recursion_limit"})
            print("  ❌ Recursion limit")
        except Exception as e:
            if "429" in str(e):
                print(f"  ⏳ Rate limit, esperando {sleep_between*2}s...")
                time.sleep(sleep_between * 2)
            results.append({"question": item["question"], "status": f"error: {e}"})

        time.sleep(sleep_between)

    return results


eval_results = run_full_eval(eval_dataset, sleep_between=8)


[1/5] ¿Cuántas lunas tiene Marte?...
  tool=0.0 | retrieval=N/A | traj=0.0 | correct=1.0 | efficient=1.0 | faithful=1.0

[2/5] ¿Quién es el CEO actual de NVIDIA?...
  tool=1.0 | retrieval=1.00 | traj=1.0 | correct=1.0 | efficient=1.0 | faithful=1.0

[3/5] ¿Cuál es la capital de Colombia?...
  tool=0.0 | retrieval=N/A | traj=0.0 | correct=1.0 | efficient=1.0 | faithful=1.0

[4/5] ¿Cuándo fue fundada la Universidad EAFIT?...
  tool=1.0 | retrieval=1.00 | traj=1.0 | correct=1.0 | efficient=1.0 | faithful=1.0

[5/5] ¿Cuál es la fórmula del agua?...
  tool=0.0 | retrieval=N/A | traj=0.0 | correct=1.0 | efficient=1.0 | faithful=1.0


## 9. Análisis de resultados

In [14]:
import pandas as pd

df = pd.DataFrame(eval_results)
ok_df = df[df["status"] == "ok"].copy()
retrieval_df = ok_df[ok_df["retrieval_score"].notna()]

# ── Tabla de resultados ────────────────────────────────────────
det_cols   = ["question", "tool_use_score", "retrieval_score", "trajectory_match"]
judge_cols = ["question", "correctness", "efficiency", "faithfulness"]

print("=" * 65)
print("EVALUADORES DETERMINÍSTICOS")
print("=" * 65)
print(ok_df[det_cols].to_string(index=False))

print("\n" + "=" * 65)
print("LLM-AS-JUDGE (3 dimensiones)")
print("=" * 65)
print(ok_df[judge_cols].to_string(index=False))

# ── Promedios ──────────────────────────────────────────────────
print("\n" + "=" * 65)
print("PROMEDIOS")
print("=" * 65)
print(f"  tool_use_score   : {ok_df['tool_use_score'].mean():.2f}  (n={len(ok_df)})")
print(f"  retrieval_score  : {retrieval_df['retrieval_score'].mean():.2f}  (n={len(retrieval_df)}, solo búsquedas)")
print(f"  trajectory_match : {ok_df['trajectory_match'].mean():.2f}  (n={len(ok_df)})")
print(f"  correctness      : {ok_df['correctness'].mean():.2f}  (n={len(ok_df)})")
print(f"  efficiency       : {ok_df['efficiency'].mean():.2f}  (n={len(ok_df)})")
print(f"  faithfulness     : {ok_df['faithfulness'].mean():.2f}  (n={len(ok_df)})")
print(f"\n  Exitosas: {len(ok_df)}/{len(df)} | Errores: {len(df[df['status'] != 'ok'])}")

EVALUADORES DETERMINÍSTICOS
                                 question  tool_use_score  retrieval_score  trajectory_match
              ¿Cuántas lunas tiene Marte?             0.0              NaN             False
       ¿Quién es el CEO actual de NVIDIA?             1.0              1.0              True
         ¿Cuál es la capital de Colombia?             0.0              NaN             False
¿Cuándo fue fundada la Universidad EAFIT?             1.0              1.0              True
            ¿Cuál es la fórmula del agua?             0.0              NaN             False

LLM-AS-JUDGE (3 dimensiones)
                                 question  correctness  efficiency  faithfulness
              ¿Cuántas lunas tiene Marte?          1.0         1.0           1.0
       ¿Quién es el CEO actual de NVIDIA?          1.0         1.0           1.0
         ¿Cuál es la capital de Colombia?          1.0         1.0           1.0
¿Cuándo fue fundada la Universidad EAFIT?          1.0      

## 10. Integración con LangSmith: dataset
LangSmith organiza las evaluaciones en **datasets** — colecciones de ejemplos
con inputs y outputs de referencia. Cada ejemplo contiene:
- `inputs`: lo que recibe el agente (la pregunta)
- `outputs`: el ground truth (expected_answer, expected_tools, expected_content)
El patrón get-or-create evita duplicar ejemplos si se corre la celda varias veces.

In [16]:
client = Client()
dataset_name = "agent-evals"

datasets = list(client.list_datasets(dataset_name=dataset_name))

if datasets:
    dataset = datasets[0]
    print(f"Dataset existente: {dataset.name}")
else:
    dataset = client.create_dataset(dataset_name)
    for item in eval_dataset:
        client.create_example(
            inputs={"question": item["question"]},
            outputs={
                "expected_answer": item["expected_answer"],
                "expected_tools": item["expected_tools"],
                "expected_content": item["expected_content"],  # ← necesario para eval_retrieval
            },
            dataset_id=dataset.id,
        )
    print(f"Dataset creado con {len(eval_dataset)} ejemplos")

Dataset creado con 5 ejemplos


## 11. Integración con LangSmith: experimento

`client.evaluate()` corre el agente sobre cada ejemplo del dataset,
aplica los evaluadores y registra los resultados en LangSmith.

Cada llamada crea un **experimento** con nombre único (prefix + hash).
Puedes comparar múltiples experimentos en el dashboard — por ejemplo,
distintos modelos o versiones del system prompt — sobre el mismo dataset.

`max_concurrency=1` es importante para no superar los rate limits de NVIDIA NIM.
Con Groq puedes subir a 2-3.

In [17]:
def agent_target(inputs: dict) -> dict:
    """Función que LangSmith llama por cada ejemplo del dataset."""
    result = run_agent_and_get_trajectory(inputs["question"])
    return {
        "answer": result["answer"],
        "trajectory": result["trajectory"],
        "tools_called": result["tools_called"],
    }

def eval_tool_use(outputs: dict, reference_outputs: dict) -> dict:
    result = evaluate_tool_use(
        outputs["tools_called"],
        reference_outputs["expected_tools"]
    )
    return {"key": "tool_use", "score": result["score"]}

def eval_retrieval(outputs: dict, reference_outputs: dict) -> dict:
    result = evaluate_retrieval(
        outputs["trajectory"],
        reference_outputs.get("expected_content", [])
    )
    if result["score"] is None:
        return {"key": "retrieval_quality", "score": None}
    return {"key": "retrieval_quality", "score": result["score"]}

def eval_trajectory(outputs: dict, reference_outputs: dict) -> dict:
    reference = build_reference_trajectory(
        outputs["trajectory"][0]["content"],
        reference_outputs["expected_tools"],
        reference_outputs["expected_answer"],
    )
    result = trajectory_evaluator(
        outputs=outputs["trajectory"],
        reference_outputs=reference,
    )
    return {"key": "trajectory_match", "score": float(result["score"])}

def eval_llm_judge(outputs: dict) -> list:
    """Una sola llamada al LLM — retorna 3 scores como lista."""
    result = multidim_judge(outputs["trajectory"])
    return [
        {"key": "correctness",  "score": result["correctness"]},
        {"key": "efficiency",   "score": result["efficiency"]},
        {"key": "faithfulness", "score": result["faithfulness"]},
    ]

experiment = client.evaluate(
    agent_target,
    data=dataset_name,
    evaluators=[eval_tool_use, eval_retrieval, eval_trajectory, eval_llm_judge],
    experiment_prefix="llama-3.1-70b",
    max_concurrency=1,
)

View the evaluation results for experiment: 'llama-3.1-70b-d514d282' at:
https://smith.langchain.com/o/8bd1c395-79be-449b-9151-f0c6da07f701/datasets/5af59c62-9895-447f-9900-68774d13dc0b/compare?selectedSessions=c6d66a91-e5d3-4952-86d2-94f94c10362e




0it [00:00, ?it/s]

## 10. Reflexión: ¿qué mide cada evaluador?

| Evaluador | Tipo | Necesita referencia | Qué mide |
|---|---|---|---|
| `tool_use` | Determinístico | Sí (`expected_tools`) | Si el agente usó las tools correctas |
| `retrieval_quality` | Determinístico | Sí (`expected_content`) | Si los resultados de búsqueda son relevantes |
| `trajectory_match` | Determinístico | Sí (trayectoria ref.) | Si el proceso fue el esperado |
| `correctness` | Probabilístico | No | ¿La respuesta final es correcta? |
| `efficiency` | Probabilístico | No | ¿Usó las tools mínimas necesarias? |
| `faithfulness` | Probabilístico | No | ¿La respuesta está fundamentada en los resultados? |

**¿Cuándo usar cada uno?**
- `tool_use`: cuando las tools correctas están bien definidas
- `trajectory_match`: cuando tienes trayectorias de referencia anotadas
- `llm_as_judge`: evaluación general sin ground truth exacto

**Limitaciones del LLM-as-judge:**
- No siempre consistente entre ejecuciones
- Puede tener sesgos del modelo juez
- Más caro (requiere otra llamada al LLM)

## 📝 Ejercicio para estudiantes

1. Agrega 3 preguntas más al dataset — incluye al menos una que requiera
   `web_search` (con `expected_content`) y una sin tools.

2. Cambia `trajectory_match_mode` de `"subset"` a `"strict"` y vuelve
   a correr la evaluación. ¿Cambian los scores? ¿Por qué?

3. El evaluador `multidim_judge` llama al LLM una vez por pregunta.
   ¿Cómo cambiarían los scores si usaras un modelo juez diferente
   (más pequeño o más grande)? Pruébalo cambiando el modelo en
   `llm.with_structured_output(EvalScores)`.

4. Actualmente `faithfulness` penaliza cuando el agente responde desde
   su conocimiento sin usar tools. ¿Es esto siempre incorrecto?
   Modifica el prompt de `CUSTOM_EVAL_PROMPT` para que solo penalice
   `faithfulness` cuando `expected_tools` no está vacío.